# The transformer block, and what depth costs

MichAl Academy, unit 4.3.

Unit 4.2 measured one attention layer on its own. This notebook builds the rest
of the block around it, one piece at a time, and every piece is introduced by a
measurement showing what goes wrong without it.

Two tasks run through it, both made here.

**The order task**, from unit 3.10. A run of filler with one `A` and one `B` in
it somewhere, and the label is which came first. Chance is 0.5.

**The two-hop task**, new here. Position 0 names one of sixteen slots, the
sixteen slots after it each hold one answer token, and the label is the token in
the named slot. Answering means two steps: read the pointer, then use it. Chance
is 1 in 16, or 0.0625.


In [ ]:
import time
import warnings

import numpy as np
import torch
from torch import nn

warnings.filterwarnings("ignore")
torch.set_num_threads(1)

SLOTS = 16
VOCAB = 2 * SLOTS + 1            # pointers, answers, and one readout token
LENGTH = SLOTS + 2


def order_task(n, length, seed):
    rng = np.random.default_rng(seed)
    X = np.zeros((n, length), dtype=np.int64)
    y = np.zeros(n, dtype=np.int64)
    for i in range(n):
        a, b = rng.choice(length, 2, replace=False)
        X[i, a], X[i, b] = 1, 2
        y[i] = 1 if a < b else 0
    return X, y


def twohop_task(n, seed):
    rng = np.random.default_rng(seed)
    X = np.zeros((n, LENGTH), dtype=np.int64)
    pointers = rng.integers(0, SLOTS, n)
    answers = rng.integers(0, SLOTS, (n, SLOTS))
    X[:, 0] = pointers                       # which slot holds the answer
    X[:, 1:1 + SLOTS] = answers + SLOTS      # the slots themselves
    X[:, -1] = 2 * SLOTS                     # the position we read the answer off
    return X, answers[np.arange(n), pointers]


X, y = twohop_task(2, 0)
print("two sequences, and the label:")
for row, label in zip(X, y):
    print(" ", row, "->", label)
print("\nThe first number names a slot, counting from zero. Slot contents are the"
      "\n16 numbers after it, each offset by 16. The last number marks the readout.")


## The block

A transformer block is an attention layer with three more things around it.

- A **feed-forward layer**: a small two-layer network applied to each position
  on its own, after the attention. Attention moves information between
  positions; nothing in it does any work on a position by itself.
- A **residual connection**: every part adds its output to its input rather than
  replacing it.
- **Layer normalisation**: each position's vector is rescaled to mean 0 and
  standard deviation 1 before each part.

Every one of them can be switched off in the class below, which is how the rest
of this notebook measures what each is for.


In [ ]:
class Block(nn.Module):
    def __init__(self, d, heads, ffn=True, residual=True, norm=True):
        super().__init__()
        self.heads, self.ffn, self.residual, self.norm = heads, ffn, residual, norm
        self.q, self.k, self.v = (nn.Linear(d, d) for _ in range(3))
        self.proj = nn.Linear(d, d)
        self.n1, self.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        if ffn:
            self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.ReLU(),
                                     nn.Linear(4 * d, d))

    def attend(self, x):
        B, L, d = x.shape
        dh = d // self.heads
        shape = (B, L, self.heads, dh)
        q = self.q(x).view(shape).transpose(1, 2)
        k = self.k(x).view(shape).transpose(1, 2)
        v = self.v(x).view(shape).transpose(1, 2)
        w = ((q @ k.transpose(-2, -1)) / dh ** 0.5).softmax(dim=-1)
        return self.proj((w @ v).transpose(1, 2).reshape(B, L, d))

    def forward(self, x):
        h = self.attend(self.n1(x) if self.norm else x)
        x = x + h if self.residual else h
        if self.ffn:
            h = self.mlp(self.n2(x) if self.norm else x)
            x = x + h if self.residual else h
        return x


class Transformer(nn.Module):
    def __init__(self, vocab, classes, length, d=32, heads=4, blocks=1,
                 positions=True, **kw):
        super().__init__()
        self.emb = nn.Embedding(vocab, d)
        self.pos = nn.Embedding(length, d) if positions else None
        self.blocks = nn.ModuleList([Block(d, heads, **kw) for _ in range(blocks)])
        self.out = nn.Linear(d, classes)

    def forward(self, x):
        h = self.emb(x)
        if self.pos is not None:
            h = h + self.pos(torch.arange(x.shape[1]))
        for b in self.blocks:
            h = b(h)
        return self.out(h[:, -1, :])


def train(model, X, y, epochs, lr=0.003, batch_size=64, seed=0):
    torch.manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    Xt, yt = torch.tensor(X), torch.tensor(y)
    last = 0.0
    for _ in range(epochs):
        order = torch.randperm(len(Xt))
        total = 0.0
        for i in range(0, len(Xt), batch_size):
            idx = order[i:i + batch_size]
            opt.zero_grad()
            loss = loss_fn(model(Xt[idx]), yt[idx])
            loss.backward()
            opt.step()
            total += float(loss) * len(idx)
        last = total / len(Xt)
    return model, last


def accuracy(model, X, y):
    model.eval()
    with torch.no_grad():
        pred = model(torch.tensor(X)).argmax(dim=1)
    model.train()
    return float((pred == torch.tensor(y)).float().mean())


def n_params(m):
    return sum(p.numel() for p in m.parameters())


print("one block, 32 wide, 4 heads:",
      n_params(Transformer(VOCAB, SLOTS, LENGTH)), "parameters")


## Order has to be added on purpose

Attention scores a query against a key. Neither carries any record of where in
the sequence it came from, so the layer sees a bag of tokens.

The order task cannot be answered from a bag: both classes contain exactly one
`A` and one `B`. The fix is a **positional embedding**, a second lookup table
indexed by position rather than by token, added to the token's embedding before
the first block.


In [ ]:
started = time.time()
print("order task, chance 0.5\n")
print("length   no positions   with positions")
for length in (12, 24):
    X_tr, y_tr = order_task(6000, length, 10 + length)
    X_te, y_te = order_task(1000, length, 50 + length)
    row = []
    for positions in (False, True):
        torch.manual_seed(0)
        model, _ = train(Transformer(3, 2, length, blocks=1, positions=positions),
                         X_tr, y_tr, epochs=20)
        row.append(accuracy(model, X_te, y_te))
    print(f"{length:6d}   {row[0]:.4f}         {row[1]:.4f}")
print(f"({time.time() - started:.1f}s)")


Without positions the model is near chance rather than exactly at it, and the
reason is worth checking rather than waving away: the answer is read off the
**last** position, so an example whose `A` or `B` happens to land there can still
be answered from the token identity alone. That is 2 sequences in 12 and 2 in 24.


In [ ]:
for length in (12, 24):
    solvable = 2 / length
    print(f"length {length}: {solvable:.3f} of examples have A or B last, "
          f"so guessing the rest predicts {solvable + (1 - solvable) * 0.5:.3f}")


## Two hops need two blocks

The two-hop task cannot be answered in one step. The readout position's query is
built from its own token, which is the same in every sequence, so a single block
attends to the same places every time and cannot condition on a pointer it has
not read yet.


In [ ]:
started = time.time()
X_tr, y_tr = twohop_task(6000, 1)
X_te, y_te = twohop_task(1000, 2)
print(f"two-hop task, chance {1 / SLOTS:.4f}\n")
for blocks in (1, 2):
    torch.manual_seed(0)
    model, loss = train(Transformer(VOCAB, SLOTS, LENGTH, blocks=blocks),
                        X_tr, y_tr, epochs=30)
    print(f"{blocks} block(s): {accuracy(model, X_te, y_te):.4f}   "
          f"{n_params(model)} parameters")
print(f"({time.time() - started:.1f}s)")


## At this depth, none of the rest matters

Here is the uncomfortable measurement. Take the two-block model that just solved
the task and remove one piece at a time.


In [ ]:
started = time.time()
print("two blocks, chance 0.0625\n")
for label, kw in [("full block     ", {}),
                  ("no feed-forward", {"ffn": False}),
                  ("no residual    ", {"residual": False}),
                  ("no layer norm  ", {"norm": False})]:
    torch.manual_seed(0)
    model, loss = train(Transformer(VOCAB, SLOTS, LENGTH, blocks=2, **kw),
                        X_tr, y_tr, epochs=30)
    print(f"{label}: {accuracy(model, X_te, y_te):.4f}   "
          f"{n_params(model):6d} parameters")
print(f"({time.time() - started:.1f}s)")


Nothing is lost, and the feed-forward-free model is much smaller. On the evidence
so far, three quarters of the block is decoration.

It is not. The evidence so far only covers two blocks.

## The same removals at twelve blocks


In [ ]:
started = time.time()
print("twelve blocks, chance 0.0625\n")
for label, kw in [("full block     ", {}),
                  ("no residual    ", {"residual": False}),
                  ("no layer norm  ", {"norm": False})]:
    torch.manual_seed(0)
    model, loss = train(Transformer(VOCAB, SLOTS, LENGTH, blocks=12, **kw),
                        X_tr, y_tr, epochs=10)
    print(f"{label}: {accuracy(model, X_te, y_te):.4f}   final loss {loss:.4f}")
print(f"({time.time() - started:.1f}s)")


## Why the residual connection is what saves it

A residual connection adds each part's output to its input. That leaves a path
from the loss back to the embedding that passes through **additions only**, so
the gradient arrives whatever the twelve blocks in between did to it.

Unit 3.4.1 measured the same failure in a deep dense network. This is the
transformer's version of it.


In [ ]:
for label, kw in [("with residual   ", {}), ("without residual", {"residual": False})]:
    torch.manual_seed(0)
    model = Transformer(VOCAB, SLOTS, LENGTH, blocks=12, **kw)
    loss = nn.CrossEntropyLoss()(model(torch.tensor(X_tr[:256])),
                                 torch.tensor(y_tr[:256]))
    loss.backward()
    print(f"{label}: gradient reaching the embedding table "
          f"{float(model.emb.weight.grad.abs().mean()):.3e}")


## Mixture of experts, in arithmetic

A mixture-of-experts layer holds several copies of the feed-forward layer and a
small router that picks which one each token goes through. The parameters
multiply; the work per token does not, because only the chosen expert runs.

Nothing needs training to see the point.


In [ ]:
d = 32
one_expert = (d * 4 * d + 4 * d) + (4 * d * d + d)
print("width 32, one expert chosen per token\n")
print("experts   parameters   used per token")
for experts in (1, 4, 16, 64):
    router = d * experts + experts
    print(f"{experts:7d}   {experts * one_expert + router:10d}   {one_expert + router:12d}")


The last row holds 64 times the parameters of the first and does 1.25 times the
work.

That is the trade a mixture of experts makes, and it is also its cost: every one
of those parameters has to be held in memory even though almost none of them run
for any given token.
